🚀 Great. You've completed:

Day 1 → Data understanding + merge + popularity baseline

Day 2 → Weighted ratings + popularity filtering

Day 3 → User-based collaborative filtering

Day 4 → Item-based collaborative filtering

Day 5 → SVD / Matrix Factorization


Now we move toward making this feel like a real recommendation system, not just isolated predictions.

# 🚀 Day 6 — Generate Actual Top-N Recommendations

Until now we did:

Predict rating for one user + one movie

Example:

model.predict(uid=1, iid=50)

returns:

4.2

But Netflix doesn't show:

Movie 50 → 4.2

It shows:

Recommended for you:

1. Star Wars
2. Titanic
3. Toy Story
4. Godfather
5. Matrix

So today we build:

Top-N recommendation engine


---

🧠 Logic

For a given user:

Step 1

Find movies already watched

Step 2

Find movies not watched

Step 3

Predict ratings for unseen movies

Step 4

Sort predictions

Step 5

Return Top N


---

Step 1 — Movies user already rated

user_id = 1

watched_movies = ratings[
    ratings['user_id']==user_id
]['movie_id'].tolist()

print("Watched:",len(watched_movies))


---

Step 2 — Get all movies

all_movies = ratings['movie_id'].unique()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:
        unwatched_movies.append(movie)

print("Unwatched:",len(unwatched_movies))


---

Step 3 — Predict ratings for unseen movies

predictions=[]

for movie in unwatched_movies:

    pred=model.predict(
        uid=user_id,
        iid=movie
    )

    predictions.append(
        (movie,pred.est)
    )


---

Step 4 — Sort predictions

predictions=sorted(
    predictions,
    key=lambda x:x[1],
    reverse=True
)


---

Step 5 — Show Top 10 recommendations

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id
    ]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


---

Expected output (example)

Star Wars (1977) ---> 4.85
Shawshank Redemption ---> 4.79
Godfather ---> 4.73
Titanic ---> 4.68
Toy Story ---> 4.61


---

🧠 What is happening now?

Before:

Predict single score

Now:

Predict all unseen movies
↓
Rank them
↓
Recommend top movies

This is the first version of a real recommendation engine.


---

🎯 Homework

1. Try different users

user_id=5
user_id=20
user_id=100

Check:

Do recommendations change?

Are some movies repeated?



---

2. Compare heavy vs light users

Check:

ratings.groupby(
    'user_id'
)['movie_id'].count()

Choose:

user with many ratings

user with few ratings


Then compare recommendation quality.

Question to think about:

Which user gets better recommendations and why?

This becomes important for the cold-start problem later.



In [1]:
# Load dataset

from surprise import Dataset, Reader, SVD, accuracy

import pandas as pd

from surprise.model_selection import train_test_split

import matplotlib.pyplot as plt

In [2]:
ratings_cols = [ 'user_id', 'movie_id', 'rating', 'timestamp' ]

ratings = pd.read_csv ( '../data/ml-100k/u.data', sep = '\t', names = ratings_cols )

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [23]:
movie_cols = [ 'movie_id', 'title' ]

movies = pd.read_csv ( r"C:\Users\Mugundhan\ML_Learning\recommendation_system\data\ml-100k\u.item", sep = '|', encoding = 'latin-1', usecols = [0, 1], names = movie_cols )

movies.head()

df = pd.merge ( ratings, movies, on = 'movie_id' )

In [3]:
# Prepare data for surprise

reader = Reader ( rating_scale = ( 1, 5 ) )  

data = Dataset.load_from_df ( ratings[ [ 'user_id', 'movie_id', 'rating' ] ], reader )

# Surprise expects ( user, item, rating )

# internally it builds -> matrices, latents vectors, embeddings 

In [4]:
# Train test split

trainset, testset = train_test_split ( data, test_size = 0.2, random_state = 42 )


In [5]:
# Train SVD model

model = SVD()

model.fit ( trainset )

# model is learning hidden user preferences, hidden movie characteristics, latent embeddings

# no direct cor relation

In [6]:
# Make Predictions

prediction = model.predict ( uid = 1, iid = 50 ) # predicting How user with id - 1 may like movie - 50

print (prediction)

user: 1          item: 50         r_ui = None   est = 4.85   {'was_impossible': False}


In [8]:
# Evaluate

predictions = model.test ( testset )

accuracy.rmse ( predictions )

# RMSE -> Root mean sq error -> Lower = better

RMSE: 0.9367


0.9366919775793281

In [9]:
# Day 8 Work started

# Logic 
#    -> Step 1 - Find Movies already rated/Watched
#    -> Step 2 - Find movies not watched
#    -> Step 3 - Predict ratings for unwatched movies
#    -> Step 4 - Sort Predictions
#    -> Step 5 - Return Top N

In [10]:

# Step 1 -> Movies users already rated

user_id = 1

watched_movies = ratings [ ratings ['user_id'] == user_id ]['movie_id'].tolist()

print ( "Watched : ", len ( watched_movies ) )

Watched :  272


In [11]:
# Step 2 -> Get all movies not watched

all_movies = ratings ['movie_id'].unique ()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:

        unwatched_movies.append ( movie )

print ( " Unwatched Movies :  ", len ( unwatched_movies ) )

 Unwatched Movies :   1410


In [15]:
# Step 3 -> Predict ratings for un watched movies

predictions = []

for movie in unwatched_movies:

    pred = model.predict ( uid = user_id, iid = movie )

    predictions.append ( ( movie, pred.est ) )

In [19]:
# Step 4 -> Sort

predictions = sorted( predictions, key = lambda x : x[1] , reverse = True )

In [26]:
# Step 5 -> Top N ( 10 ) recommendation sys

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


To Kill a Mockingbird (1962) ---> 4.85
Rear Window (1954) ---> 4.80
North by Northwest (1959) ---> 4.80
Third Man, The (1949) ---> 4.69
Vertigo (1958) ---> 4.57
Close Shave, A (1995) ---> 4.57
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963) ---> 4.54
Sunset Blvd. (1950) ---> 4.54
Secrets & Lies (1996) ---> 4.53
Wings of Desire (1987) ---> 4.51


# Home Work

In [29]:
# Try different users

user_id = 50

watched_movies = ratings [ ratings [ 'user_id' ] == user_id ]['movie_id'].tolist()

print ( " watched Movies : ", len ( watched_movies ) )

all_movies = ratings [ 'movie_id' ].unique()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:

        unwatched_movies.append ( movie )

print ( " Unwatched Movies :  ", len ( unwatched_movies ) )

predictions = []

for movie in unwatched_movies:

    pred = model.predict ( uid = user_id, iid = movie )
    
    predictions.append ( ( movie, pred.est ) )

# Step 4 -> Sort

predictions = sorted( predictions, key = lambda x : x[1] , reverse = True )

# Step 5 -> Top N ( 10 ) recommendation sys

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


 watched Movies :  24
 Unwatched Movies :   1658
Third Man, The (1949) ---> 4.79
Close Shave, A (1995) ---> 4.70
Wrong Trousers, The (1993) ---> 4.69
Princess Bride, The (1987) ---> 4.61
Shawshank Redemption, The (1994) ---> 4.55
Treasure of the Sierra Madre, The (1948) ---> 4.53
Star Wars (1977) ---> 4.52
To Kill a Mockingbird (1962) ---> 4.52
Maltese Falcon, The (1941) ---> 4.51
Rear Window (1954) ---> 4.51


In [30]:
# Try different users

user_id = 12

watched_movies = ratings [ ratings [ 'user_id' ] == user_id ]['movie_id'].tolist()

print ( " watched Movies : ", len ( watched_movies ) )

all_movies = ratings [ 'movie_id' ].unique()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:

        unwatched_movies.append ( movie )

print ( " Unwatched Movies :  ", len ( unwatched_movies ) )

predictions = []

for movie in unwatched_movies:

    pred = model.predict ( uid = user_id, iid = movie )
    
    predictions.append ( ( movie, pred.est ) )

# Step 4 -> Sort

predictions = sorted( predictions, key = lambda x : x[1] , reverse = True )

# Step 5 -> Top N ( 10 ) recommendation sys

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


 watched Movies :  51
 Unwatched Movies :   1631
12 Angry Men (1957) ---> 5.00
Shawshank Redemption, The (1994) ---> 4.96
Close Shave, A (1995) ---> 4.95
Rear Window (1954) ---> 4.92
Casablanca (1942) ---> 4.88
Usual Suspects, The (1995) ---> 4.87
Thin Man, The (1934) ---> 4.85
Vertigo (1958) ---> 4.82
Wallace & Gromit: The Best of Aardman Animation (1996) ---> 4.79
Manchurian Candidate, The (1962) ---> 4.77


In [45]:
# Compare Heavy vs Light users

ratings_count = ratings.groupby ( 'user_id' )[ 'movie_id' ].count()

sorted_users = ratings_count.sort_values ( ascending = False )

print ( " Sorted_users \n ", sorted_users )

heavy_user = sorted_users.index [0]

heavy_count = sorted_users.iloc [0]

light_user = sorted_users.index [-1]

light_count = sorted_users.iloc [-1]

print ( f" Heavy users : { heavy_user } heavy count : { heavy_count } " )

print ( f" light users : { light_user } light count : { light_count } " )


 Sorted_users 
  user_id
405    737
655    685
13     636
450    540
276    518
      ... 
685     20
475     20
36      20
732     20
596     20
Name: movie_id, Length: 943, dtype: int64
 Heavy users : 405 heavy count : 737 
 light users : 596 light count : 20 


In [46]:
# Try different users - light user

user_id = light_user

watched_movies = ratings [ ratings [ 'user_id' ] == user_id ]['movie_id'].tolist()

print ( " watched Movies : ", len ( watched_movies ) )

all_movies = ratings [ 'movie_id' ].unique()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:

        unwatched_movies.append ( movie )

print ( " Unwatched Movies :  ", len ( unwatched_movies ) )

predictions = []

for movie in unwatched_movies:

    pred = model.predict ( uid = user_id, iid = movie )
    
    predictions.append ( ( movie, pred.est ) )

# Step 4 -> Sort

predictions = sorted( predictions, key = lambda x : x[1] , reverse = True )

# Step 5 -> Top N ( 10 ) recommendation sys

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


 watched Movies :  20
 Unwatched Movies :   1662
Shawshank Redemption, The (1994) ---> 4.73
Casablanca (1942) ---> 4.71
Close Shave, A (1995) ---> 4.69
Schindler's List (1993) ---> 4.64
Raiders of the Lost Ark (1981) ---> 4.56
Third Man, The (1949) ---> 4.50
North by Northwest (1959) ---> 4.49
Princess Bride, The (1987) ---> 4.48
Usual Suspects, The (1995) ---> 4.45
Manchurian Candidate, The (1962) ---> 4.44


In [47]:
# Try different users - Heavy user

user_id = heavy_user

watched_movies = ratings [ ratings [ 'user_id' ] == user_id ]['movie_id'].tolist()

print ( " watched Movies : ", len ( watched_movies ) )

all_movies = ratings [ 'movie_id' ].unique()

unwatched_movies = []

for movie in all_movies:

    if movie not in watched_movies:

        unwatched_movies.append ( movie )

print ( " Unwatched Movies :  ", len ( unwatched_movies ) )

predictions = []

for movie in unwatched_movies:

    pred = model.predict ( uid = user_id, iid = movie )
    
    predictions.append ( ( movie, pred.est ) )

# Step 4 -> Sort

predictions = sorted( predictions, key = lambda x : x[1] , reverse = True )

# Step 5 -> Top N ( 10 ) recommendation sys

top10=predictions[:10]

for movie_id,rating in top10:

    movie_name=movies[
        movies['movie_id']==movie_id]['title'].values[0]

    print(
        f"{movie_name} ---> {rating:.2f}"
    )


 watched Movies :  737
 Unwatched Movies :   945
Gaslight (1944) ---> 3.55
It's a Wonderful Life (1946) ---> 3.36
Breakfast at Tiffany's (1961) ---> 3.24
Best Men (1997) ---> 3.20
Toy Story (1995) ---> 3.16
Some Folks Call It a Sling Blade (1993) ---> 3.13
Fargo (1996) ---> 3.12
Mr. Smith Goes to Washington (1939) ---> 3.09
His Girl Friday (1940) ---> 3.09
When We Were Kings (1996) ---> 3.05


Excellent observations again 🔥 You found two important things:

1. Why different users still get many of the same movies


2. Your heavy/light user code has a logic issue



Let's take them one at a time.


---

1️⃣ Why are different users getting many of the same movies?

You expected:

User12 → completely different list
User50 → completely different list

but got:

Shawshank Redemption
Rear Window
Close Shave
...

appearing repeatedly.

At first glance this looks wrong, but the reason is actually understandable.


---

Why this happens

Remember what SVD learned:

user features + movie features

Some movies are simply:

globally very strong movies

Examples:

Shawshank Redemption

Star Wars

Casablanca

Rear Window


These movies have:

many ratings

high average ratings

strong positive movie embeddings


So even if users differ, the model may still think:

"Most people probably like these"


---

Think of Netflix:

Two users:

User A → likes action
User B → likes thrillers

Netflix may still recommend:

Breaking Bad
Dark

to both users because they are generally highly rated.

Only the ranking score changes:

User 12:

Shawshank → 4.96

User 50:

Shawshank → 4.55

Meaning:

Both may like it,
but User12 probably likes it more

So personalization exists, but not extreme personalization yet.


---

Why this happens more in MovieLens100k

MovieLens100k:

100k ratings
943 users
1682 movies

This is relatively small.

So SVD sometimes leans heavily toward:

movie popularity + movie bias

instead of learning extremely unique preferences.

In larger datasets:

Netflix

YouTube

Amazon


recommendations become much more personalized.


---

2️⃣ Heavy/Light user code issue

Your code:

ratings_count = ratings.groupby('user_id')['movie_id'].count()

sorted_ratings_count = sorted(ratings_count, reverse=True)

heavy_users = sorted_ratings_count[0]
light_users = sorted_ratings_count[-1]

Problem:

You're sorting the counts only, not keeping the user IDs attached.

After sorting:

[737,685,600,520,...]

You lost:

UserID → Count relationship

So:

heavy_users

becomes:

737

which is number of movies watched, not user ID.


---

Correct code

ratings_count = ratings.groupby(
    'user_id'
)['movie_id'].count()

# Sort while keeping user ids
sorted_users = ratings_count.sort_values(
    ascending=False
)

heavy_user = sorted_users.index[0]
heavy_count = sorted_users.iloc[0]

light_user = sorted_users.index[-1]
light_count = sorted_users.iloc[-1]

print(
    f"Heavy User : {heavy_user}"
)

print(
    f"Movies watched : {heavy_count}"
)

print()

print(
    f"Light User : {light_user}"
)

print(
    f"Movies watched : {light_count}"
)

Expected:

Heavy User : 405
Movies watched : 737

Light User : 202
Movies watched : 20

(IDs may differ.)


---

Now use:

user_id = heavy_user

and later:

user_id = light_user

Generate recommendations for both.

Observe:

Heavy user

Many watched movies
→ more preference information
→ stronger personalization

Light user

Few watched movies
→ weak preference information
→ recommendations become generic


---

This is another real-world issue:

Less user history
↓
Less personalization

which is the beginning of the cold-start problem.

